## Read the data

In [6]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"

# 1. Read data (Now it is the same data)

In [7]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
# DATA_PATH_BASE = 'D:/Google Drive/Work/ITMO/data' #windows
# DATA_PATH_BASE = '/home/denklewer/Data' #ubuntu

DATA_PATH = f'{DATA_PATH_BASE}/length_dependent_hmm/per_length_per_kfold_split/' # work laptop
PATH_TO_SAVE_RESULTS = f'{DATA_PATH_BASE}/length_dependent_hmm/peptide_cores/'
APPROACH = "DECOY_PATHS"

In [8]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = list(sorted(get_available_alleles(DATA_PATH)))

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'
          ]
ALLELES = [
    'HLA-A*01:01',
    'HLA-A*03:01',
    # 'HLA-A*02:01б'
   'HLA-DRB1*01:01',
    'HLA-DPA1*01:03/DPB1*04:01'
          ]
selected_alleles = ALLELES
ALLELES

['HLA-A*01:01', 'HLA-A*03:01', 'HLA-DRB1*01:01', 'HLA-DPA1*01:03/DPB1*04:01']

## 2.0 Merge alleles together

In [9]:
# combination_size = len(ALLELES)
ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'])
# ALLELES = sorted([ 'HLA-A*03:01'])
ALLELES = sorted([   'HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03/DPB1*04:01'
                      ])
# ALLELES = sorted(['HLA-DPA1*01:03-DPB1*04:01'])
#
# ALLELES = sorted(['HLA-DRB1*01:01', 'HLA-DPA1*01:03-DPB1*04:01'])
#

combination_size = 1
TOTAL_SPLITS = 1
TARGET_LENGTHS= [12, 13, 14, 15, 16, 17, 18]
# TARGET_LENGTHS= [8, 9, 10, 11, 12]
RUNS = 15
USE_WEIGHTS = True

# SELECTED_RUN = 0
# SELECTED_RUN = {0,1,2,3,4,5, 6, 7, 8, 9}
SELECTED_RUN = {0,1}

# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0
models_to_read = ['v']

## Merge train data together (multiple alleles to single set of peptides)

In [10]:
from collections import defaultdict
global_selected_alleles = ALLELES # Data alleles we selected (but the number will be reduced by acceptable alleles)

selected_alleles = global_selected_alleles


per_allele_representations_train = dict()
per_allele_representations_test = dict()
per_name_representation_train = dict()
per_name_representation_test = dict()

def handle_intersections(per_allele_per_kfold_per_length_peptides):
    per_allele_result_peptide_sets =  defaultdict(set)
    per_allele_per_length_peptide_sets =  defaultdict(set)
    PER_LENGTH_SIZE = {target_length: min([len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])) for allele_name in selected_alleles]) for target_length in TARGET_LENGTHS}
    print("Min unique sizes", PER_LENGTH_SIZE)
    for target_length in TARGET_LENGTHS:
        print(f"Length {target_length}")
        for allele_name in selected_alleles:
            if target_length in per_allele_per_kfold_per_length_peptides[allele_name][0]:
                print(f"initial length for  allele { allele_name} TL {target_length} is {len(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])}")
                print(f"unique length for  allele { allele_name} TL {target_length} is {len(set(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length]))}")
                unique_peptides = np.unique(per_allele_per_kfold_per_length_peptides[allele_name][0][target_length])
                if "imm" not in allele_name: #do not subsambple for imm
                    peptides = set(np.random.choice(unique_peptides, size=PER_LENGTH_SIZE[target_length], replace=False))
                else:
                    peptides = set(unique_peptides)
                print(f"Selected/Subsampled {len(peptides)} peptides")
                per_allele_per_length_peptide_sets[allele_name] = peptides
        # get all intersections
        intersections = list()
        for i in range(len(selected_alleles)):
            for j in range(i + 1, len(selected_alleles)):
                set1 = per_allele_per_length_peptide_sets[selected_alleles[i]]
                set2 = per_allele_per_length_peptide_sets[selected_alleles[j]]
                intersect = set1.intersection(set2)
                print(f"{selected_alleles[i]} intersect {selected_alleles[j]} - {len(intersect)}")
                intersections.append(intersect)
        # delete all intersections
        for allele_name, peptide_set in per_allele_per_length_peptide_sets.items():
            print(f"Allele {allele_name}: deleted", end=" ")
            initial_length = len(peptide_set)
            for intersect in intersections:
                peptide_set = peptide_set.difference(intersect)
            result_length = len(peptide_set)
            print(f" {initial_length - result_length} of intersected peptides")
            per_allele_result_peptide_sets[allele_name] = per_allele_result_peptide_sets[allele_name].union(peptide_set)
    return per_allele_result_peptide_sets

print("Train")
per_allele_peptide_train_sets = handle_intersections(per_allele_per_kfold_per_length_binders_train)
print("Test")
per_allele_peptide_test_sets = handle_intersections(per_allele_per_kfold_per_length_binders_test)



train_peptides_b = set(set().union(*per_allele_peptide_train_sets.values()))

test_peptides_b = set(set().union(*per_allele_peptide_test_sets.values()))

print(f"Train {len(train_peptides_b)}, Test {len(test_peptides_b)}")



data_df = pd.DataFrame(index=list(train_peptides_b))
for allele_name in selected_alleles:
    data_df.loc[list(per_allele_peptide_train_sets[allele_name]), 'allele'] = allele_name



Train
Min unique sizes {12: 342, 13: 763, 14: 1173, 15: 5076, 16: 1352, 17: 939, 18: 619}
Length 12
initial length for  allele HLA-DRB1*01:01 TL 12 is 354
unique length for  allele HLA-DRB1*01:01 TL 12 is 342
Selected/Subsampled 342 peptides
Allele HLA-DRB1*01:01: deleted  0 of intersected peptides
Length 13
initial length for  allele HLA-DRB1*01:01 TL 13 is 812
unique length for  allele HLA-DRB1*01:01 TL 13 is 763
Selected/Subsampled 763 peptides
Allele HLA-DRB1*01:01: deleted  0 of intersected peptides
Length 14
initial length for  allele HLA-DRB1*01:01 TL 14 is 1248
unique length for  allele HLA-DRB1*01:01 TL 14 is 1173
Selected/Subsampled 1173 peptides
Allele HLA-DRB1*01:01: deleted  0 of intersected peptides
Length 15
initial length for  allele HLA-DRB1*01:01 TL 15 is 5315
unique length for  allele HLA-DRB1*01:01 TL 15 is 5076
Selected/Subsampled 5076 peptides
Allele HLA-DRB1*01:01: deleted  0 of intersected peptides
Length 16
initial length for  allele HLA-DRB1*01:01 TL 16 is 143

In [11]:
from data_reading_methods import merge_per_allele_sets_into_combination_sets

per_allele_per_kfold_per_length_binders_train_old = per_allele_per_kfold_per_length_binders_train
per_allele_per_kfold_per_length_binders_test_old = per_allele_per_kfold_per_length_binders_test

per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test = merge_per_allele_sets_into_combination_sets(
    per_allele_per_kfold_per_length_binders_train, per_allele_per_kfold_per_length_binders_test,
    combination_size=combination_size,
    selected_alelles=ALLELES,
    total_splits=TOTAL_SPLITS
)
OLD_ALLELES = ALLELES
NEW_ALLELES = list(per_allele_per_kfold_per_length_binders_test.keys())
ALLELES = NEW_ALLELES

HLA-DRB1*01:01
Length 8: HLA-DRB1*01:01 - 39 
Length 9: HLA-DRB1*01:01 - 109 
Length 10: HLA-DRB1*01:01 - 121 
Length 11: HLA-DRB1*01:01 - 165 
Length 12: HLA-DRB1*01:01 - 354 
Length 13: HLA-DRB1*01:01 - 812 
Length 14: HLA-DRB1*01:01 - 1248 
Length 15: HLA-DRB1*01:01 - 5315 
Length 16: HLA-DRB1*01:01 - 1433 
Length 17: HLA-DRB1*01:01 - 981 
Length 18: HLA-DRB1*01:01 - 651 
Length 19: HLA-DRB1*01:01 - 305 
Length 20: HLA-DRB1*01:01 - 396 
Length 21: HLA-DRB1*01:01 - 157 
Length 22: HLA-DRB1*01:01 - 76 
Length 23: HLA-DRB1*01:01 - 58 
Length 24: HLA-DRB1*01:01 - 64 
Length 25: HLA-DRB1*01:01 - 46 
Length 26: HLA-DRB1*01:01 - 12 
Length 27: HLA-DRB1*01:01 - 11 
Length 28: HLA-DRB1*01:01 - 7 
Length 30: HLA-DRB1*01:01 - 6 


In [12]:
selected_alleles

['HLA-DRB1*01:01']

In [13]:
data_df.allele.value_counts()

HLA-DRB1*01:01    10264
Name: allele, dtype: int64

# Read models

In [21]:
MODELS_DESCRIPTIONS = {
    "viterbi_simple_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 7,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 7,
        "core_end_state": 8

    },
    "viterbi_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [35],
        "core_end_state": [36]

    },
    "viterbi_multiple_7": {
        "path": "experiment_results/core_multiple_allele/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-DRB1*01:01',
                    'HLA-DPA1*01:03/DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [35, 36],
        "core_end_state": [37, 38],
        "need_to_delete": [39, 40, 41, 42]

    }



    # "mix_of_alleles_balanced":
    # {
    #     "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
    #     "path": "experiment_results/alleles_with_groups_balanced/",
    #     "states": 25,
    #     "params_string": "5-min_iters",
    #     "target_lengths": [9, 10]
    # }
}

In [22]:
selected_models = [
     "viterbi_with_pseudo_and_weights_7"
]


In [23]:
from hmm_logic_methods import load_model

In [24]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_binding_model = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    model_alleles = set(model_alleles).intersection(set(OLD_ALLELES))
    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_binding_model[model_name] =  load_model(name_to_model_root[model_name], model_name)

## Mark important states in model(will be used for core call)

In [25]:
import logomaker
from collections import defaultdict
def probs_to_info_matrix(distribution):
    amino_acids = list(distribution.parameters[0].keys())
    dummy_df = pd.DataFrame(columns = sorted(amino_acids), index=[0],  dtype=float)
    for i, (acid, prob) in enumerate(distribution.parameters[0].items()):
        dummy_df.loc[0, acid] = prob
    dummy_df.index.name = "pos"
    info_matrix = logomaker.transform_matrix(dummy_df, from_type='probability', to_type='information')
    return info_matrix

per_model_total_df = dict()

for model_key, model in per_name_binding_model.items():
    per_state_information = dict()
    per_state_distribution = dict()
    per_level_information = defaultdict(float)

    for state in model.states:
        if state not in [model.start, model.end]:
            info_matrix = probs_to_info_matrix(state.distribution)
            per_state_distribution[state.name] = state.distribution.parameters[0]
            per_state_information[state.name] = sum(info_matrix.loc[0])
            # per_level_information[name_to_level[state.name]] += per_state_information[state.name]
    info_df = pd.DataFrame(per_state_information.items(), columns=["state", "info"])
    distr_df = pd.DataFrame(per_state_distribution.values())
    total_df = pd.concat([info_df, distr_df], axis=1)
    per_model_total_df[model_key] = total_df

In [26]:
filtered_dfs = list()
per_model_important_states = dict()
if models_to_read == ['bw']:
    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df[total_df['info'] >= 1]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)

if models_to_read == ['v']:
    per_model_anchors = dict()
    per_name_models = per_name_binding_model

    core_start_states= MODELS_DESCRIPTIONS[selected_models[0]]['core_start_state']
    core_end_states = MODELS_DESCRIPTIONS[selected_models[0]]['core_end_state']

    for i, (model_key, total_df) in enumerate(per_model_total_df.items()):
        total_df = total_df.sort_values(by='info', ascending=False).copy()
        total_df = total_df.loc[core_start_states + core_end_states]
        total_df['model'] = i
        total_df['model_name'] = model_key
        filtered_dfs.append(total_df)
        per_model_important_states[model_key] = list(total_df['state'].values)





## Calculate core guesses

In [27]:
# prepare peptides for viterbi calculation (expected to be unique)
allele_peptides = data_df.index.values


len(allele_peptides)

10264

In [28]:
TEST_PDB_CORES = False
if TEST_PDB_CORES:
    df=pd.read_csv('test/binding_cores', sep='\t')
    df= df[~df['Antigen'].str.contains('X')]
    df = df[df['Allele'] == 'DRB1*01:01']

    allele_peptides = df.Antigen.values
    true_labels=df['Core (PDB)'].values
    netmhcpan = df['Predicted core.5'].values

In [29]:
from hmm_logic_methods import calculate_model_core_guesses
selected_peptides = [
    # "APEEILAEKSFNTVPY", 'EALTFSREHGPGVAEPE',
    #                  'YYSLLMPILTLTRAL',
    #                      'SYLTNMFATWSP',
    #                  'NPLIRHENRMVLAST',
    #                      'KPPKPVSKMRMATPLLM',
                         ]
total_df, selected_peptides_to_paths = calculate_model_core_guesses(allele_peptides, per_name_models, per_model_important_states,
                                    selected_peptides
                                   )

Model 0_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 10264/10264 [00:01<00:00, 8178.21it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 10264 


100%|██████████| 10264/10264 [00:00<00:00, 21483.48it/s]


Model 1_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 10264/10264 [00:01<00:00, 9122.87it/s]


... (50 outputs left out)

Model 14_run-12-13-14-15-16-17-18-35-st-viterbi-alg-30-min_iters_model-HLA_DRB1_01_01-0
Get viterbi paths for each peptide...


100%|██████████| 10264/10264 [00:01<00:00, 6156.61it/s]


Subselect unique paths (if unique path argument passed)..
	Unique paths: 10264 


100%|██████████| 10264/10264 [00:00<00:00, 14262.66it/s]


In [30]:
data_df = data_df.join(total_df)


In [31]:
data_df.head()

,allele,simplified_path,peptide
FLCLFLLPSLATVAY,HLA-DRB1*01:01,"[0.0, 0.0, 3.0300000000000002, 15.319999999999...",FLCLFLLPSLATVAY
KPLALYAFSKRSQVIK,HLA-DRB1*01:01,"[1.05, 4.25, 4.25, 5.3100000000000005, 6.35000...",KPLALYAFSKRSQVIK
FASFEAQGALANIA,HLA-DRB1*01:01,"[1.03, 1.03, 3.04, 15.279999999999998, 15.2799...",FASFEAQGALANIA
KDPQTNQWKLGEPLELRR,HLA-DRB1*01:01,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.01, 15.01, 15...",KDPQTNQWKLGEPLELRR
NLSNVLATITTGVLDI,HLA-DRB1*01:01,"[0.0, 0.0, 0.0, 1.07, 2.12, 15.55, 15.55, 15.5...",NLSNVLATITTGVLDI


In [32]:
data_df.allele.value_counts()

HLA-DRB1*01:01    10264
Name: allele, dtype: int64

In [33]:
threshold = 7.57 #threshold value for core call

In [34]:
data_df['core_start'] = data_df['simplified_path'].apply(lambda x: np.where(np.array(x) > threshold)[0]).apply(lambda x: x[0] if len(x) > 0 else 0)

data_df['core_end'] = data_df['simplified_path'].apply(lambda x: np.where(np.array(x) > threshold)[0]).apply(lambda x: x[-1] if len(x)> 0 else 1)
data_df['core_len'] = data_df['core_end'] - data_df['core_start'] + 1
data_df['peptide_len'] = data_df.index.str.len()
data_df['core_peptide'] =data_df.apply(lambda row: row.name[row['core_start']:row['core_end'] + 1], axis=1)
data_df = data_df[~data_df.index.duplicated(keep='first')]
data_df['simplified_path'] = data_df['simplified_path'].apply(lambda x: np.round(x.astype(float),2))
data_df['core_path'] = data_df.apply(lambda row: row['simplified_path'][row['core_start']:row['core_end'] + 1] , axis=1)


In [35]:
data_df.core_len.value_counts()

9     10254
10       10
Name: core_len, dtype: int64

In [36]:
data_df.head()

,allele,simplified_path,peptide,core_start,core_end,core_len,peptide_len,core_peptide,core_path
FLCLFLLPSLATVAY,HLA-DRB1*01:01,"[0.0, 0.0, 3.03, 15.32, 15.32, 15.32, 15.32, 1...",FLCLFLLPSLATVAY,3,11,9,15,LFLLPSLAT,"[15.32, 15.32, 15.32, 15.32, 15.32, 15.32, 15...."
KPLALYAFSKRSQVIK,HLA-DRB1*01:01,"[1.05, 4.25, 4.25, 5.31, 6.35, 15.81, 15.81, 1...",KPLALYAFSKRSQVIK,5,13,9,16,YAFSKRSQV,"[15.81, 15.81, 15.81, 15.81, 14.76, 11.56, 11...."
FASFEAQGALANIA,HLA-DRB1*01:01,"[1.03, 1.03, 3.04, 15.28, 15.28, 15.28, 15.28,...",FASFEAQGALANIA,3,11,9,14,FEAQGALAN,"[15.28, 15.28, 15.28, 15.28, 15.28, 15.28, 14...."
KDPQTNQWKLGEPLELRR,HLA-DRB1*01:01,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.01, 15.01, 15...",KDPQTNQWKLGEPLELRR,7,15,9,18,WKLGEPLEL,"[15.01, 15.01, 15.01, 15.01, 15.01, 15.01, 15...."
NLSNVLATITTGVLDI,HLA-DRB1*01:01,"[0.0, 0.0, 0.0, 1.07, 2.12, 15.55, 15.55, 15.5...",NLSNVLATITTGVLDI,5,13,9,16,LATITTGVL,"[15.55, 15.55, 15.55, 15.55, 15.55, 15.55, 15...."


# Save core data to files

In [37]:
import os
if not os.path.exists(PATH_TO_SAVE_RESULTS):
    os.makedirs(PATH_TO_SAVE_RESULTS)
data_df.to_csv("{}extractd_cores.csv".format(PATH_TO_SAVE_RESULTS,),sep=";", index=True)